In [ ]:
# In PostgreSQL, create a database named "poc_to_prod" and a user named "XYZ" with password "********". 
# Grant all privileges on the database to the user.

In [ ]:
# Create a new schema with the following SQL command:
# CREATE SCHEMA IF NOT EXISTS memory;

In [ ]:
# Enable the extensions in the "poc_to_prod" database:

# CREATE EXTENSION IF NOT EXISTS vector;
# CREATE EXTENSION IF NOT EXISTS "uuid-ossp";
# CREATE EXTENSION IF NOT EXISTS "pg_trgm";


In [ ]:
# While creating the table, the very basic columns of the table would be as follows:
# MESSAGE_ID 
# SENDER (HUMAN OR ASSISTANT)
# MESSAGE 
# CREATED_AT (TIMESTAMP)

In [ ]:
# If I only add these columns, then I would not be able to do a lot of things. 
# For example, I would not be able to get messages only for a specific user. Obviously i dont want users to see each other's messages.
# So I would need to add a USER_ID column to the table.
# 
# The same user might have multiple conversations with the assistant. 
# So I would need to add a SESSION_ID column to the table and manage the conversations like sessions.
# 
# From this, we have understood that the relational design of the memory would have user_id & session_id along with message_id.


In [ ]:
# But what else should we include in the memory?
# What should our memory design look like?
# What are the cases where this simplistic design would fail?

In [ ]:
# Schema Design :

# users -> user_id (PK), name, email, password, created_at, updated_at, last_login_at
# sessions -> session_id (PK), user_id (FK), session_name, created_at, terminated_at
# messages -> message_id (PK), session_id (FK), sender, message, created_at

# This design allows us to manage users, their sessions, and the messages within those sessions.
# Let us proceed with this design and implement it in our database.

In [ ]:
# memory.users
"""
CREATE TABLE MEMORY.USERS (
    user_id UUID PRIMARY KEY DEFAULT UUID_GENERATE_V4(),
    name VARCHAR(255),
    email VARCHAR(255),
    password TEXT NOT NULL,
    created_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    last_login_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""

In [ ]:
# memory.sessions
"""
CREATE TABLE MEMORY.SESSIONS (
    session_id UUID PRIMARY KEY DEFAULT UUID_GENERATE_V4(),
    user_id UUID NOT NULL,
    session_name VARCHAR(60),
    is_active BOOLEAN DEFAULT TRUE,
    created_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    terminated_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""

In [ ]:
# memory.chats
"""
CREATE TABLE MEMORY.CHATS (
    chat_id UUID PRIMARY KEY DEFAULT UUID_GENERATE_V4(),
    session_id UUID NOT NULL,
    sender TEXT NOT NULL CHECK (sender IN ('user', 'assistant')),
    message TEXT NOT NULL,
    created_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""

In [ ]:
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "poc_to_prod",
    "user": "postgres",
    "password": "**********"
}

In [ ]:
import psycopg2
from psycopg2.extras import RealDictCursor

In [ ]:
conn = psycopg2.connect(**DB_CONFIG)
conn.autocommit = False
cur = conn.cursor(cursor_factory=RealDictCursor)
print("Connected:", conn.status)  # 1 = CONNECTION_OK

In [ ]:
# In case you got error while connecting for authentication,
# just check username and password.
# You will get the username from PGadmin dashboard and password is the one that you have set while creating the user in PostgreSQL.
# You can also update password for the user in PGadmin dashboard if you have forgotten it with this query:
# ALTER USER postgres WITH PASSWORD '**********'; -- replace postgres with your username and ********** with your new password.

In [ ]:
query = "SELECT * FROM MEMORY.USERS;"
cur.execute(query)
users = cur.fetchall()

In [ ]:
users

In [ ]:
# Don't forget to close the connection after you're done. This is important to free up resources and avoid potential memory leaks.
cur.close()
conn.close()

In [ ]:
# Let us write following codes :
# 1. Create a new user in the database.
# 2. Create a new session for that user.
# 3. Add a few messages to that session.
# 4. Fetch all / "N" messages for that session. This will be the conversation history that we will use as memory for the assistant to refer to while answering the user's queries.

# Basically we need to implement the Create (INSERT) and Read (SELECT) operations for the three tables that we have created. 
# We can implement Update and Delete operations as well but for now, let's focus on Create and Read operations.

In [ ]:
# 1. Create a new user in the database.

query_create_user = """
INSERT INTO MEMORY.USERS (name, email, password)
VALUES (%s, %s, %s)
RETURNING user_id, name, email, created_at;
"""

In [ ]:
def create_user(name, email, password):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    cur.execute(query_create_user, (name, email, password))
    new_user = cur.fetchone()
    conn.commit()  # Commit the transaction to save changes to the database
    cur.close()
    conn.close()
    return new_user

In [ ]:
# As per data governance and security best practices, we should not store plain text passwords in the database.
# We should hash the passwords before storing them. We can use libraries like bcrypt or argon2 for hashing passwords in Python.
# For example, we can use bcrypt library to hash the password before storing it in the database. Here is how you can do it:

import bcrypt

def hash_password(plain_password):
    # Generate a salt and hash the password
    salt = bcrypt.gensalt()
    hashed_password = bcrypt.hashpw(plain_password.encode('utf-8'), salt)
    return hashed_password.decode('utf-8')  # Store as string in the database

In [ ]:
# Create a new user with hashed password

create_user("Pritesh Jha", "pritesh-jha@email.com", hash_password("my_secure_password"))

* Check in pgadmin console. The new user must be added there.

![added_new_user.png](../assets/snapshots/added_new_user.png)

In [ ]:
# Create a new session for that user

In [ ]:
# For creating a new session, we need to have the user_id of the user for whom we want to create the session.
# We can get the user_id from the result of create_user function or read it using a select query with email.

In [ ]:
select_user_query = """
SELECT user_id FROM MEMORY.USERS WHERE email = %s;
"""


In [ ]:
def get_user_id_by_email(email):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    cur.execute(select_user_query, (email,))
    user = cur.fetchone()
    cur.close()
    conn.close()
    return user['user_id'] if user else None

In [ ]:
user_id = get_user_id_by_email("pritesh-jha@email.com")
print("User ID:", user_id)

In [ ]:
# Once we have the user_id, we can create a new session for that user using the following query:

create_session_query = """
INSERT INTO MEMORY.SESSIONS (user_id, session_name)
VALUES (%s, %s)
RETURNING session_id, user_id, session_name, created_at;
"""

In [ ]:
def create_session(user_id, session_name):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    cur.execute(create_session_query, (user_id, session_name))
    new_session = cur.fetchone()
    conn.commit()  # Commit the transaction to save changes to the database
    cur.close()
    conn.close()
    return new_session

In [ ]:
create_session(user_id, "My First Session")

* Check in pgadmin :
  
!["add_new_session.png"](../assets/snapshots/add_new_session.png)

In [ ]:
# Now that we have created a session, we can add a few messages to that session using the following query:

add_message_query = """
INSERT INTO MEMORY.CHATS (session_id, sender, message)
VALUES (%s, %s, %s)
RETURNING chat_id, session_id, sender, message, created_at;
"""

In [ ]:
# The previous query can be used to add messages from both the user and the assistant. 
# We can differentiate between the sender by using the sender column. 
# For example, we can use "user" for messages sent by the user and "assistant" for messages sent by the assistant.

# For assistant, let us us the functions from src/chat_service.py to generate responses 
# and then store those responses in the database with sender as "assistant".


In [ ]:
import sys
import os

# Add the parent directory to the system path to import from src module
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.chat_service import ChatService

In [ ]:
from src.core.models import LLMConfig, ChatConfig

# Initialize configurations
llm_config = LLMConfig(
    provider="ollama",  # or "openai"
    model="llama3.1:latest",  # or your desired model
    temperature=0.3,
    base_url="http://localhost:11434"  # for ollama
)

chat_config = ChatConfig(
    system_prompt="You are a helpful assistant. Be concise and clear in your responses."
)

# Initialize ChatService
chat = ChatService(
    llm_config=llm_config,
    chat_config=chat_config
)

In [ ]:
def add_message_to_db(session_id, sender, message):
    """Add a message to the database.
    
    Args:
        session_id: The session ID
        sender: The sender ("user" or "assistant")
        message: The message content
        
    Returns:
        The inserted message record
    """
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    cur.execute(add_message_query, (session_id, sender, message))
    inserted_message = cur.fetchone()
    conn.commit()
    cur.close()
    conn.close()
    return inserted_message

In [ ]:
# First, get the session_id from the previous session we created
# You can query it or use the one created earlier

get_session_query = """
SELECT session_id FROM MEMORY.SESSIONS WHERE user_id = %s ORDER BY created_at DESC LIMIT 1;
"""

def get_latest_session(user_id):
    """Get the latest session for a user."""
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    cur.execute(get_session_query, (user_id,))
    session = cur.fetchone()
    cur.close()
    conn.close()
    return session['session_id'] if session else None

In [ ]:
# Get the session_id
user_id_str = str(user_id)
session_id_str = str(get_latest_session(user_id_str))
print(f"Using latest session: {session_id_str}")

In [ ]:
# Example: Chat with the assistant
# This will:
# 1. Add message to the database
# 2. Get response from the assistant
# 3. Add the assistant's response to the database


In [ ]:
# 1. Add user message to database

user_message = "What is machine learning?"
user_msg_record = add_message_to_db(session_id_str, "user", user_message)


In [ ]:
print(f"User message added: {user_msg_record}")


In [ ]:
# 2. Get response from the assistant
print("\nGenerating assistant response...")
assistant_response = chat.get_response(user_message)
print(f"Assistant response: {assistant_response}\n")

In [ ]:
#3.  Add assistant message
assistant_msg_record = add_message_to_db(session_id_str, "assistant", assistant_response)
print(f"ASSISTANT: {assistant_response}\n")

* Add messages to MEMORY.CHATS

!["add_messages.png](../assets/snapshots/add_messages.png)

In [ ]:
# The above code demonstrates how to add messages to the database and generate responses from the assistant.
# Lets make one functions that does the 3 steps mentioned above in one go, 
# so that we can have a cleaner code and just call that function whenever we want to have a conversation with the assistant.

In [ ]:
def chat_with_assistant(session_id, user_message):
    """Convenient function to chat with assistant and store messages in database.
    
    This function:
    1. Adds the user message to the database
    2. Gets response from the LLM assistant
    3. Adds the assistant response to the database
    
    Args:
        session_id: The session ID
        user_message: The user's message
        
    Returns:
        A tuple of (user_message_record, assistant_message_record)
    """
    # Add user message
    user_msg_record = add_message_to_db(session_id, "user", user_message)
    print(f"USER: {user_message}")
    
    # Get assistant response
    assistant_response = chat.get_response(user_message)
    
    # Add assistant message
    assistant_msg_record = add_message_to_db(session_id, "assistant", assistant_response)
    print(f"ASSISTANT: {assistant_response}\n")
    
    return user_msg_record, assistant_msg_record

In [ ]:
chat_with_assistant(session_id_str, "Can you explain Relu activation function in deep learning?")

In [ ]:
# Fetch conversation history for the session
fetch_conversation_query = """
SELECT chat_id, sender, message, created_at 
FROM MEMORY.CHATS 
WHERE session_id = %s
ORDER BY created_at ASC;
"""

def get_conversation_history(session_id):
    """Get complete conversation history for a session.
    
    Args:
        session_id: The session ID
        
    Returns:
        List of message records
    """
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    cur.execute(fetch_conversation_query, (session_id,))
    messages = cur.fetchall()
    cur.close()
    conn.close()
    return messages

In [ ]:
# Get and display conversation history
conversation = get_conversation_history(session_id_str)

In [ ]:
conversation